In [1]:
import requests, zipfile
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

from sklearn.metrics import accuracy_score, precision_score, recall_score

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip"

response = requests.get(url)
if response.status_code == 200:
    with open("spam.zip", "wb") as f:
        f.write(response.content)
    with zipfile.ZipFile("spam.zip") as z:
        z.extractall(".")
else:
    print("Download failed")

df = pd.read_csv("SMSSpamCollection", sep="\t", header=None, names=["label", "message"])
df["label_num"] = df["label"].map({"ham": 0, "spam": 1})

X_train, X_test, y_train, y_test = train_test_split(
    df["message"], df["label_num"], test_size=0.2, random_state=42
)


vectorizer = CountVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)


model = MultinomialNB(alpha=0.5)
model.fit(X_train_vec, y_train)


y_pred = model.predict(X_test_vec)

print("===== Overall =====")
print("Accuracy  :", round(accuracy_score(y_test, y_pred) * 100, 2), "%")

print("\n===== HAM (Not Spam) =====")
print("Precision :", round(precision_score(y_test, y_pred, pos_label=0) * 100, 2), "%")
print("Recall    :", round(recall_score(y_test, y_pred, pos_label=0) * 100, 2), "%")

print("\n===== SPAM =====")
print("Precision :", round(precision_score(y_test, y_pred, pos_label=1) * 100, 2), "%")
print("Recall    :", round(recall_score(y_test, y_pred, pos_label=1) * 100, 2), "%")

# ham messages
messages = [
    "Hey, I got a free movie ticket from my office event, want to join me?",
    "Congrats on your win in the college competition! Let's celebrate later.",
    "Call me when you reach, I have an important offer to discuss about the project.",
    "You won the match today, great job! Dinner is on me.",
    "Free snacks available in the lab today, come fast before it finishes!",
    "Reminder: You have won a chance to present your idea tomorrow in class.",
    "Limited seats left for the workshop, register soon if you're interested.",
    "Your account project submission has been approved, good work!",
    "Hey, there's a special offer at the canteen today, let's go together.",
    "Urgent: Please call me back, it's about tomorrow's exam schedule."
]


for msg in messages:
    pred = model.predict(vectorizer.transform([msg]))[0]
    result = "SPAM - This message is spam." if pred == 1 else "NOT SPAM - This message is not spam."
    print(f"Message  : {msg}")
    print(f"Result   : {result}\n")

ModuleNotFoundError: No module named 'pandas'